# Create HAL config and shutter files

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config,
)
from MERci.acquisition.display import print_frame_table, display_xml
from MERci.visualization       import visualize_shutter_sequence

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE = "MF3"

# ── Auto-detect HAL config template ────────────────────────────────────
# Looks for hal-config-*{MICROSCOPE}*.xml in MERci/data/configs/hal/
# (case-insensitive match on the microscope name)
_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(
        f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}"
    )
HAL_TEMPLATE = _hal_candidates[0]

# ── Imaging file format and camera settings ────────────────────────────
# FILE_TYPE options: ".zarr" (default), ".dax", ".tiff"
FILE_TYPE     = ".zarr"
EXPOSURE_TIME = 0.25      # seconds

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"HAL template : {HAL_TEMPLATE.name}")

## create hal config for bits

In [ ]:
# ── Define imaging sequence ────────────────────────────────────────────
z_bead    = 0
z_min     = 1
z_max     = 25
z_step    = 1
z_pos     = np.arange(z_min, z_max+1, z_step) 
bead_seq  = [488]
color_seq = [750,    650,    560]
end_seq   = [488]

# Scan mode — controls the order in which z-planes and colors are acquired:
#   "interleaved" : all colors acquired at each Z-nanopositioner position before stepping to the next z
#                   (AOTF / fast electronic switching)
#   "sequential"  : full z-stack acquired per color with boustrophedon Z-nanopositioner sweep,
#                   then switch to the next color (physical shutter / slow switching)
SCAN_MODE = "interleaved"

frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                               microscope=MICROSCOPE, scan_mode=SCAN_MODE)
name        = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / f"frame_table_{name}.csv"
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file ──────────────────────────────────────────────────
shutter_name = f"shutter-{name}.xml"
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ────────────────────────────────────────────────────
hal_name   = f"hal-config-{MICROSCOPE.lower()}-bits-{name}.xml"
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

## create hal config for cells

In [ ]:
# ── Define imaging sequence ────────────────────────────────────────────
z_bead    = 0
z_min     = 1
z_max     = 25
z_step    = 1
z_pos     = np.arange(z_min, z_max+1, z_step) 
bead_seq  = [488]
color_seq = [405]
end_seq   = [488]

# Scan mode (see bits block for description; typically "interleaved" for a single-color round)
SCAN_MODE = "interleaved"

frame_table = get_frame_table(z_bead, bead_seq, color_seq, end_seq, z_pos,
                               microscope=MICROSCOPE, scan_mode=SCAN_MODE)
name        = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Color sequence name: {name}")
print_frame_table(frame_table)

# ── Save frame table ────────────────────────────────────────────────────
ft_path = METADATA_DIR / f"frame_table_{name}.csv"
frame_table.to_csv(ft_path)
print(f"Frame table saved: {ft_path}")

# ── Write shutter file ──────────────────────────────────────────────────
shutter_name = f"shutter-{name}.xml"
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path)
print(f"Shutter file saved: {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ────────────────────────────────────────────────────
hal_name   = f"hal-config-{MICROSCOPE.lower()}-cells-{name}.xml"
hal_output = SETTINGS_DIR / hal_name
create_hal_config(HAL_TEMPLATE, frame_table, shutter_name, hal_output,
                  file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME)
print(f"HAL config saved: {hal_output}")
display_xml(hal_output)

# ── Visualise shutter sequence ──────────────────────────────────────────
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)